<a href="https://colab.research.google.com/github/sbravosol/REN235/blob/main/Copy_of_rusle_ren237.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este será el proyecto base que usaremos para crear un modelo RUSLE usando Google Earth Engine

In [1]:
import math
import os

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

**Pasos para activar su cuenta de Google con un proyecto GEE**

1. Crear cuenta de Google (si no tienen una)

2. Ir a https://earthengine.google.com/signup/ e iniciar sesión con su cuenta de Google. Completar el formulario de registro seleccionando 'Uso no comercial / académico'
3. Crear un proyecto en Google Cloud Console
Ir a https://console.cloud.google.com/projectcreate, ponerle un nombre al proyecto (ej. 'clase-erosion-nombre') y anotar ese Project ID
4. Habilitar la API de Earth Engine en el proyecto
Con el proyecto creado y seleccionado, ir a https://console.cloud.google.com/apis/library/earthengine.googleapis.com y hacer clic en 'Habilitar'
5. Registrar el proyecto para uso con Earth Engine
Ir a https://console.cloud.google.com/earth-engine/configuration y seguir los indicando que es para investigación/uso académico no comercial.


In [2]:
GEE_PROJECT = 'cuenca-reloca'

import ee

print("Autenticando...")
ee.Authenticate()

print(f"Inicializando proyecto: {GEE_PROJECT}...")
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✓ Google Earth Engine inicializado correctamente!")

    info = ee.Number(1).getInfo()
    print("✓ Conexión verificada.")

except ee.EEException as e:
    print(f"\n✗ Falló la inicialización: {e}")
    if "not registered" in str(e):
        print(f"\nACCIÓN REQUERIDA: El proyecto '{GEE_PROJECT}' no está registrado en Earth Engine.")
        print(f"1. Regístralo aquí: https://console.cloud.google.com/earth-engine/configuration?project={GEE_PROJECT}")
        print("2. O actualiza la variable GEE_PROJECT con un Project ID válido.")
    elif "permission" in str(e).lower():
        print(f"\nACCIÓN REQUERIDA: Puede que no tengas permisos para el proyecto '{GEE_PROJECT}'.")
        print("Revisa la configuración de IAM en Google Cloud Console.")
except Exception as e:
    print(f"\n✗ Ocurrió un error inesperado: {e}")

Autenticando...
Inicializando proyecto: cuenca-reloca...


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


✓ Google Earth Engine inicializado correctamente!
✓ Conexión verificada.


In [3]:
# Leemos una cuenca del inventario de erosion de CIREN
# Yo elegi la del Estero Cholqui https://inventarioerosion.ciren.cl/layers/geonode_data:geonode:r13_168_rep_1ha
import geopandas as gpd

url = "https://inventarioerosion.ciren.cl/geoserver/ows?service=WFS&version=1.0.0&request=GetFeature&typename=geonode%3Ar07_066_rea_1ha&outputFormat=json&srs=EPSG%3A32719&srsName=EPSG%3A32719"
gp_df = gpd.read_file(url)
gp_df_simple = gp_df.dissolve() # opcion de geopandas para solo tener el contorno de la cuenca
area_de_interes = geemap.geopandas_to_ee(gp_df_simple) # de geopandas a gee


In [4]:
# Visualizamos el area de interes
mapa_simple = geemap.Map()
mapa_simple.centerObject(area_de_interes, zoom=12)
mapa_simple.addLayer(area_de_interes, {'color': 'black'}, 'Área de interés')
mapa_simple

Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…

Vamos a comenzar con el primer componente de RUSLE: erosividad de la lluvia.Según Bonilla y Vidal 2011, R = 0.0028P^1.534, donde P es la precipitación anual en mm. Vamos a cargar las precipitaciones de un año hidrológico (inicia 1 de abril y termina el 31 de marzo) solo como ejemplo para recordar el concepto. Pero por lo general los estudios son anuales (1 enero a 1 de diciembre)


In [5]:
# 1. Precipitación acumulada del año hidrológico (CHIRPS diario, ~5,5 km)
#    El año hidrológico chileno va del 1 de abril al 31 de marzo.
precipitacion = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                 .filterBounds(area_de_interes)
                 .filterDate('2023-04-01', '2024-04-01')
                 .select('precipitation'))

P_anual = precipitacion.sum().clip(area_de_interes).rename('P_mm')

# 2. R = 0.028 * P^1.534   (Bonilla & Vidal 2011) -> MJ mm ha-1 h-1 an-1
R = P_anual.pow(1.534).multiply(0.028).rename('R') ##preguntar que de donde sale esta formul

# 3. Traer los valores desde el servidor a Python
est_R = R.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes.geometry(),
    scale=5566,                # resolución nativa de CHIRPS
    maxPixels=1e9,
    bestEffort=True).getInfo()

print("Rango de R:", {k: round(v) for k, v in est_R.items()})

# 4. Visualizar con el rango tomado de los propios datos
viz_R = {'min': est_R['R_min'], 'max': est_R['R_max'],
         'palette': ['white', 'yellow', 'orange', 'red']}

mapa_simple.addLayer(R, viz_R, 'Erosividad R')
mapa_simple

Rango de R: {'R_max': 690, 'R_min': 457}


Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…

Ahora vamos a descargar datos de suelo (arena, limo y arcilla + materia orgánica y usar la fórmula de EPIC para obtener valores de K. Los obtendremos de OpenLandMap https://stac.openlandmap.org/organic.carbon_usda.6a1c/collection.json?.language=en

In [6]:

# Carbono orgánico del suelo (g/kg, capa 0 cm)
carbono_organico = ee.Image('OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Arcilla (% en masa, capa 0 cm)
arcilla = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Arena (% en masa, capa 0 cm)
arena = ee.Image('OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Limo (% en masa, capa 0 cm)
limo = ee.Image(100).subtract(arena).subtract(arcilla)


In [7]:
carbono_organico_pct = carbono_organico.multiply(0.5)  # OpenLandMap: b0 x 5 g/kg → %

sn1 = ee.Image(1).subtract(arena.divide(100))  # fraccion que no es arena

# f(csand) = 0.2 + 0.3*exp[-0.0256*arena*(1 - limo/100)]
factor1 = ee.Image(0.2).add(
    ee.Image(0.3).multiply(
        arena.multiply(-0.0256).multiply(
            ee.Image(1).subtract(limo.divide(100))
        ).exp()
    )
)

# f(cl-si) = (limo / (arcilla + limo))^0.3
factor2 = (limo.divide(arcilla.add(limo))).pow(0.3)

# f(orgc) = 1 - 0.25*orgC / (orgC + exp[3.72 - 2.95*orgC])
factor3 = ee.Image(1).subtract(
    ee.Image(0.25).multiply(carbono_organico_pct).divide(
        carbono_organico_pct.add(
            (ee.Image(3.72).subtract(carbono_organico_pct.multiply(2.95))).exp()
        )
    )
)

# f(hisand) = 1 - 0.7*SN1 / (SN1 + exp[-5.51 + 22.9*SN1])
factor4 = ee.Image(1).subtract(
    ee.Image(0.7).multiply(sn1).divide(
        sn1.add((ee.Image(-5.51).add(sn1.multiply(22.9))).exp())
    )
)

# K primero se define, despues se reduce
K_us = factor1.multiply(factor2).multiply(factor3).multiply(factor4)  # unidades EPIC
K = K_us.multiply(0.1317).rename('K')   # a SI, para calzar con el R de Bonilla & Vidal

stats_K = K.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes.geometry(),
    scale=250,              # resolucion nativa de OpenLandMap
    maxPixels=1e9,
    bestEffort=True
).getInfo()

print("Rango de K en SI:", {k: round(v, 4) for k, v in stats_K.items()})

viz_K = {'min': stats_K['K_min'], 'max': stats_K['K_max'],
         'palette': ['white', 'lightgreen', 'green', 'darkgreen']}
mapa_simple.addLayer(K, viz_K, 'Erodabilidad K')
mapa_simple

Rango de K en SI: {'K_max': 0.0293, 'K_min': 0.0208}


Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…

---
## 6. Factores L y S — Topografía

$$L = \left(\frac{\lambda}{22.13}\right)^m \qquad S = \frac{0.43 + 0.30\,s + 0.043\,s^2}{6.613}$$

con $\lambda$ = longitud de pendiente (m) y $s$ = pendiente (%). El 22,13 es la longitud de la parcela estándar de la USLE (72,6 pies), así que $L = 1$ cuando la ladera mide exactamente eso.

### El exponente $m$

$m$ controla **qué tan rápido crece la erosión al alargarse la ladera**, y no es constante: depende de la pendiente misma. La razón es física. En laderas tendidas domina la erosión entre surcos (*interrill*), producida por el impacto de la gota de lluvia, que ocurre de manera pareja en toda la superficie y por lo tanto casi no depende del largo. En laderas empinadas el flujo se concentra en surcos (*rill*) que ganan volumen y velocidad ladera abajo, de modo que cada metro adicional aporta más erosión que el anterior.

Wischmeier & Smith (1978) tabularon $m$ según la razón entre erosión en surcos y entre surcos:

| Pendiente $s$ | $m$ | Proceso dominante |
|---|---|---|
| < 1 % | 0,2 | *interrill* — el largo casi no importa |
| 1 – 3 % | 0,3 | transición |
| 3 – 4,5 % | 0,4 | transición |
| > 4,5 % | 0,5 | *rill* — el largo importa mucho |


In [8]:
# --- Mapa de L ---

# 1. Pendiente a partir del DEM (recortar DESPUES, para no dañar el borde)
dem = ee.Image('USGS/SRTMGL1_003')
pendiente_pct = (ee.Terrain.slope(dem)
                 .multiply(3.14159265 / 180).tan().multiply(100)
                 .clip(area_de_interes).rename('pendiente_pct'))

# 2. L = (lambda / 22.13)^m   con lambda = tamaño de pixel
TAMANO_PIXEL_L = 30.0

m = (ee.Image(0.5)
     .where(pendiente_pct.lte(1.0), 0.2)
     .where(pendiente_pct.gt(1.0).And(pendiente_pct.lte(3.0)), 0.3)
     .where(pendiente_pct.gt(3.0).And(pendiente_pct.lte(4.5)), 0.4)
     .where(pendiente_pct.gt(4.5), 0.5)
     .rename('m'))

lam = pendiente_pct.multiply(0).add(TAMANO_PIXEL_L)
L = lam.divide(22.13).pow(m).rename('L')

# 3. Traer los valores desde el servidor a Python
est_L = L.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes.geometry(),
    scale=30,                  # resolución nativa de SRTM
    maxPixels=1e9,
    bestEffort=True).getInfo()

print("Rango de L:", {k: round(v, 3) for k, v in est_L.items()})

# 4. Visualizar con el rango tomado de los propios datos
viz_L = {'min': est_L['L_min'], 'max': est_L['L_max'],
         'palette': ['white', 'lightblue', 'blue', 'darkblue']}

mapa_simple.addLayer(L, viz_L, 'Longitud de pendiente L')
mapa_simple

Rango de L: {'L_max': 1.164, 'L_min': 1.063}


Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…

In [9]:
# --- Mapa de S ---

# 2. S = (0.43 + 0.30*s + 0.043*s^2) / 6.613
S = (pendiente_pct.pow(2).multiply(0.043)
     .add(pendiente_pct.multiply(0.30))
     .add(0.43)
     .divide(6.613)
     .rename('S'))

# 3. Traer los valores desde el servidor a Python
#    Se pide tambien el percentil 98: el maximo de S es un valor extremo aislado
est_S = S.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.percentile([98]), sharedInputs=True),
    geometry=area_de_interes.geometry(),
    scale=30,
    maxPixels=1e9,
    bestEffort=True).getInfo()

print("Rango de S:", {k: round(v, 2) for k, v in est_S.items()})

# 4. Visualizar cortando en el percentil 98 para no saturar la paleta
viz_S = {'min': est_S['S_min'], 'max': est_S['S_p98'],
         'palette': ['white', 'yellow', 'orange', 'darkred']}

mapa_simple.addLayer(S, viz_S, 'Inclinación S')
mapa_simple

Rango de S: {'S_max': 115.33, 'S_min': 0.07, 'S_p98': 22.7}


Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…


De Jong (1994) relacionó el factor $C$ de campo con el NDVI:

$$C = 0.431 - 0.805 \cdot NDVI$$


In [10]:
# --- Mapa de C ---

# 1. Compuesto Landsat 8 del año hidrológico
l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(area_de_interes)
      .filterDate('2023-04-01', '2024-04-01')
      .filter(ee.Filter.lt('CLOUD_COVER', 30)))

print("Escenas Landsat 8 disponibles:", l8.size().getInfo())

compuesto = (l8.median()
             .select(['SR_B5', 'SR_B4'])
             .multiply(0.0000275).add(-0.2)   # escala Collection 2 Level 2
             .clip(area_de_interes))

# 2. C = 0.431 - 0.805 * NDVI   (De Jong 1994), acotado a [0, 1]
ndvi = compuesto.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
C = ee.Image(0.431).subtract(ndvi.multiply(0.805)).clamp(0, 1).rename('C')

# 3. Traer los valores desde el servidor a Python
est_C = ndvi.addBands(C).reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=area_de_interes.geometry(),
    scale=30,                  # resolución nativa de Landsat
    maxPixels=1e9,
    bestEffort=True).getInfo()

print("Rango de NDVI:", {k: round(v, 3) for k, v in est_C.items() if k.startswith('NDVI')})
print("Rango de C:   ", {k: round(v, 3) for k, v in est_C.items() if k.startswith('C')})

# 4. Visualizar. NDVI con escala fija (-1 a 1 es su rango teórico);
#    C con el rango de los propios datos.
viz_ndvi = {'min': 0, 'max': 0.8, 'palette': ['brown', 'yellow', 'lightgreen', 'darkgreen']}
viz_C = {'min': est_C['C_min'], 'max': est_C['C_max'],
         'palette': ['darkgreen', 'yellow', 'orange', 'brown']}

mapa_simple.addLayer(ndvi, viz_ndvi, 'NDVI', False)   # apagada por defecto
mapa_simple.addLayer(C, viz_C, 'Cobertura C')
mapa_simple

Escenas Landsat 8 disponibles: 21
Rango de NDVI: {'NDVI_max': 0.938, 'NDVI_mean': 0.669, 'NDVI_min': -0.588}
Rango de C:    {'C_max': 0.904, 'C_mean': 0.019, 'C_min': 0}


Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…

Factor P: Prácticas de conservación

Sin un catastro de prácticas de conservación, se asigna un valor de $P$ por clase de cobertura de suelo MODIS, siguiendo Chuenchum et al. (2019). **Chuenchum, P., Xu, M., & Tang, W. (2019).** Estimation of soil erosion and sediment yield in the Lancang–Mekong River using the modified RUSLE. *Water*, 12(1), 135.https://www.mdpi.com/2073-4441/12/1/135


In [11]:
# --- Mapa de P ---

# 1. Cobertura de suelo MODIS (500 m), imagen mas reciente disponible
VALORES_P = {
    1: 0.8, 2: 0.8, 3: 0.8, 4: 0.8, 5: 0.8,    # bosques
    6: 0.8, 7: 0.8, 8: 0.8, 9: 0.8, 10: 0.8,   # matorrales, sabanas, praderas
    11: 1.0,                                    # humedales
    12: 0.5,                                    # cultivos
    13: 1.0,                                    # urbano
    14: 0.5,                                    # mosaico cultivo / vegetacion natural
    15: 1.0, 16: 1.0, 17: 1.0,                  # nieve, suelo desnudo, agua
}

mcd12 = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1')
ultima = ee.Image(mcd12.sort('system:time_start', False).first())
print("Año de la cobertura MODIS:", ee.Date(ultima.get('system:time_start')).format('YYYY').getInfo())

cobertura = ultima.clip(area_de_interes).rename('lulc')

# 2. P por clase de cobertura (Chuenchum et al. 2019)
expresion = ": ".join(f"(b('lulc') == {c}) ? {v}" for c, v in VALORES_P.items()) + ": 1.0"
P = cobertura.expression(expresion).rename('P')

# 3. Traer los valores desde el servidor a Python
est_P = P.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=area_de_interes.geometry(),
    scale=500,                 # resolucion nativa de MODIS
    maxPixels=1e9,
    bestEffort=True).getInfo()

print("Rango de P:", {k: round(v, 3) for k, v in est_P.items()})

# Superficie por clase de cobertura presente en la cuenca.
conteo = ee.Image.pixelArea().divide(1e4).addBands(cobertura).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='clase'),
    geometry=area_de_interes.geometry(), scale=500,
    maxPixels=1e9, bestEffort=True).getInfo()

NOMBRES_IGBP = {
    1: 'Bosque siempreverde aciculifolio', 2: 'Bosque siempreverde latifoliado',
    3: 'Bosque caducifolio aciculifolio', 4: 'Bosque caducifolio latifoliado',
    5: 'Bosque mixto', 6: 'Matorral cerrado', 7: 'Matorral abierto',
    8: 'Sabana arbolada', 9: 'Sabana', 10: 'Pradera', 11: 'Humedal',
    12: 'Cultivo', 13: 'Urbano', 14: 'Mosaico cultivo/natural',
    15: 'Nieve y hielo', 16: 'Suelo desnudo', 17: 'Agua',
}

print(f"\n{'Clase':>6}  {'Cobertura':<32}{'P':>6}{'ha':>12}{'%':>8}")
total = sum(g['sum'] for g in conteo['groups'])
for g in sorted(conteo['groups'], key=lambda x: -x['sum']):
    clase = int(g['clase'])
    print(f"{clase:>6}  {NOMBRES_IGBP.get(clase, '?'):<32}"
          f"{VALORES_P.get(clase, 1.0):>6}{g['sum']:>12,.0f}{100*g['sum']/total:>7.1f}%")

# 4. Visualizar. P es discreto: solo toma los valores 0.5, 0.8 y 1.0,
#    por eso la escala va fija y no derivada de los datos.
paleta_igbp = ['05450a','086a10','54a708','78d203','009900','c6b044','dcd159',
               'dade48','fbff13','b6ff05','27ff87','c24f44','a5a5a5','ff6d4c',
               '69fff8','f9ffa4','1c0dff']

mapa_simple.addLayer(cobertura, {'min': 1, 'max': 17, 'palette': paleta_igbp},
                     'Cobertura MODIS', False)
mapa_simple.addLayer(P, {'min': 0.5, 'max': 1.0,
                         'palette': ['green', 'yellow', 'red']}, 'Prácticas P')
mapa_simple

Año de la cobertura MODIS: 2024
Rango de P: {'P_max': 1, 'P_mean': 0.802, 'P_min': 0.8}

 Clase  Cobertura                            P          ha       %
     2  Bosque siempreverde latifoliado    0.8      16,279   37.9%
     8  Sabana arbolada                    0.8       9,729   22.7%
     1  Bosque siempreverde aciculifolio   0.8       8,621   20.1%
     9  Sabana                             0.8       5,981   13.9%
    10  Pradera                            0.8       1,741    4.1%
    11  Humedal                            1.0         230    0.5%
     5  Bosque mixto                       0.8         142    0.3%
     4  Bosque caducifolio latifoliado     0.8         101    0.2%
    17  Agua                               1.0          92    0.2%
     7  Matorral abierto                   0.8           3    0.0%


Map(center=[-35.678774646032366, -72.35934030677808], controls=(WidgetControl(options=['position', 'transparen…